# Week 3 — Memory-Bound Bottlenecks: FlashAttention and Serving Systems

**Course:** High-Performance Computing & Scaling Large Models  
**Practical:** vLLM (PagedAttention) integration and inference benchmarking.

---

## Learning Objectives

1. Derive the *I/O complexity* of standard attention and identify why it is memory-bound.
2. Explain the **FlashAttention** algorithm: tiling, online softmax, recomputation in the backward pass.
3. Use PyTorch's `scaled_dot_product_attention` (which dispatches to FlashAttention) and measure the speedup.
4. Understand the **KV cache** in autoregressive decoding and the *memory fragmentation* problem it creates.
5. Apply **vLLM** with PagedAttention to serve a small LLM and benchmark its throughput.


## 1. The Memory Wall in Attention

Standard scaled dot-product attention computes:

$$
\text{Attn}(Q, K, V) = \text{softmax}\!\left(\tfrac{QK^\top}{\sqrt{d}}\right) V
$$

with $Q, K, V \in \mathbb{R}^{N \times d}$. The naive implementation materializes the intermediate matrix $S = QK^\top \in \mathbb{R}^{N \times N}$ in HBM, applies softmax, then multiplies by $V$.

**Memory cost.** Materializing $S$ requires $O(N^2)$ bytes. For $N = 8192$ in FP16, this is 128 MB — *per attention head*. Multi-head attention with 32 heads pushes this past 4 GB.

**Bandwidth cost.** Each token's row of $S$ is written, read, written, and read again across the softmax and matmul-with-$V$ steps. The kernel performs $O(N^2)$ work but moves $O(N^2)$ data — arithmetic intensity is $O(1)$. It is, by the roofline classification from Week 1, *severely memory-bound*.

This is not a hypothetical inefficiency. On an A100, a naive attention kernel at $N = 4096$ achieves around 30 TFLOPs/s — less than 10 % of the FP16 peak.


## 2. FlashAttention: Tiling Without Materialization

Dao et al. (2022) observed that, although the $N \times N$ matrix $S$ appears unavoidable, the *output* of attention is only $N \times d$. If we could compute the output *block by block*, never holding the full $S$ in HBM, the bandwidth cost would drop from $O(N^2)$ to $O(N \, d)$.

The obstacle is softmax. The normalization

$$
\text{softmax}(s)_i = \dfrac{\exp(s_i)}{\sum_j \exp(s_j)}
$$

requires the *global* sum in the denominator. We cannot finish a block without seeing every other block.

### The Online Softmax Trick

FlashAttention applies a numerically stable, *streaming* reformulation. For each new block of keys, maintain a running maximum $m$ and a running denominator $\ell$:

$$
m^{\text{new}} = \max(m^{\text{old}}, m_{\text{block}})
$$

$$
\ell^{\text{new}} = e^{m^{\text{old}} - m^{\text{new}}} \, \ell^{\text{old}}
                  + e^{m_{\text{block}} - m^{\text{new}}} \, \ell_{\text{block}}
$$

The output accumulator $O$ is similarly rescaled by $e^{m^{\text{old}} - m^{\text{new}}}$ before adding the new block's contribution. At the end, $O / \ell$ yields the exact softmax-attention output — to machine precision, no approximation.

### Result

- HBM accesses: $O(N \, d \, \sqrt{N \, d / M})$, where $M$ is the size of on-chip SRAM. In practice, **5–10× less HBM traffic** than the standard algorithm.
- Wall-clock speedup on long contexts: 2–4× forward, 3–5× backward (with recomputation).
- Memory usage: $O(N)$ instead of $O(N^2)$, enabling much longer contexts.

PyTorch 2.0+ ships FlashAttention via `torch.nn.functional.scaled_dot_product_attention`. Let us measure the impact.


In [ ]:
import math, time
import torch
import torch.nn as nn
import torch.nn.functional as F

assert torch.cuda.is_available(), "This notebook requires a CUDA device."
device = torch.device("cuda")
torch.manual_seed(0)
print(f"PyTorch  : {torch.__version__}")
print(f"Device   : {torch.cuda.get_device_name(0)}")
print(f"SDPA backends available:")
print(f"  - flash      : {torch.backends.cuda.flash_sdp_enabled()}")
print(f"  - mem-eff    : {torch.backends.cuda.mem_efficient_sdp_enabled()}")
print(f"  - math       : {torch.backends.cuda.math_sdp_enabled()}")


In [ ]:
# Naive reference attention (for comparison)
def naive_attention(Q, K, V, mask=None):
    d = Q.size(-1)
    scores = (Q @ K.transpose(-2, -1)) / math.sqrt(d)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
    attn = scores.softmax(dim=-1)
    return attn @ V


def bench(fn, *args, n_iter=20, warmup=5):
    for _ in range(warmup):
        _ = fn(*args)
    torch.cuda.synchronize()
    s = torch.cuda.Event(enable_timing=True); e = torch.cuda.Event(enable_timing=True)
    s.record()
    for _ in range(n_iter):
        _ = fn(*args)
    e.record()
    torch.cuda.synchronize()
    return s.elapsed_time(e) / n_iter


In [ ]:
# Compare naive vs SDPA across context lengths.
B, H, D = 2, 16, 64
print(f"{'seq_len':>8} | {'naive (ms)':>12} {'mem (MB)':>10} | "
      f"{'SDPA  (ms)':>12} {'speedup':>8}")
print("-" * 70)

for L in (512, 1024, 2048, 4096, 8192):
    Q = torch.randn(B, H, L, D, device=device, dtype=torch.float16)
    K = torch.randn(B, H, L, D, device=device, dtype=torch.float16)
    V = torch.randn(B, H, L, D, device=device, dtype=torch.float16)

    # Naive may OOM at very long contexts — guard it.
    try:
        torch.cuda.reset_peak_memory_stats()
        ms_naive = bench(naive_attention, Q, K, V)
        mem_naive = torch.cuda.max_memory_allocated() / 1e6
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        ms_naive, mem_naive = float('nan'), float('nan')

    ms_sdpa = bench(F.scaled_dot_product_attention, Q, K, V)
    speedup = (ms_naive / ms_sdpa) if ms_naive == ms_naive else float('nan')

    print(f"{L:>8} | {ms_naive:>12.3f} {mem_naive:>10.1f} | "
          f"{ms_sdpa:>12.3f} {speedup:>8.2f}×")


**Reading the table.**

- The naive kernel's peak memory grows quadratically. Around $L = 8192$ on an A100-40GB, it OOMs.
- SDPA's memory grows *linearly*, allowing context lengths that were previously infeasible.
- Speedups grow with sequence length — at $L=4096$, expect 3–5× on Ampere.

Numerical agreement: the FlashAttention output is *exact* (to FP16 precision), unlike approximate attention methods such as Performer or Linformer.


## 3. The KV Cache: Inference's Hidden Cost

In autoregressive generation, each new token attends to *all previous tokens*. Without caching, generating $L$ tokens performs $\sum_{t=1}^L \Theta(t \, d) = \Theta(L^2 \, d)$ work — quadratic. The standard remedy is the **KV cache**: store $K_t$ and $V_t$ for every generated token, so token $t+1$ does $\Theta(t \, d)$ work — linear.

The cost is memory. For a model with $L_{\text{ctx}}$ context length, $H$ heads, $d$ head dim, $n_\ell$ layers, and a batch of $B$ requests, the KV cache occupies

$$
2 \cdot B \cdot L_{\text{ctx}} \cdot H \cdot d \cdot n_\ell \cdot \text{sizeof(dtype)}
$$

For a 7B model (32 layers, 32 heads, $d = 128$, FP16) with $L_{\text{ctx}} = 4096$, the per-request KV cache is **~2 GB**. For a batch of 16 concurrent requests, **32 GB** — exceeding the weight memory of the model itself.

### The Fragmentation Problem

Real serving traffic has *heterogeneous* sequence lengths: one user sends a 100-token prompt, another sends 3000. A static allocation per request — at the maximum context length — wastes most of the memory. Naive dynamic allocation suffers from external fragmentation: free blocks scattered across the address space, none large enough for an incoming request.

**vLLM's PagedAttention** (Kwon et al., 2023) borrows the virtual-memory paging idea from operating systems. The KV cache is divided into fixed-size *blocks* (e.g., 16 tokens each), with a per-request *block table* mapping logical token positions to physical blocks. Blocks are allocated on demand, freed when a request finishes, and may be shared across requests with identical prefixes (system-prompt sharing).

Empirically: 2–4× higher throughput than naive serving, and near-100 % GPU memory utilization.


## 4. KV Cache: A From-Scratch Implementation

Before reaching for vLLM, let us implement the simplest possible KV cache to understand what PagedAttention actually solves.


In [ ]:
class CausalSelfAttention(nn.Module):
    """A minimal causal self-attention layer with an optional KV cache."""
    def __init__(self, d=128, n_heads=4, max_len=2048):
        super().__init__()
        assert d % n_heads == 0
        self.n_heads = n_heads
        self.head_dim = d // n_heads
        self.qkv = nn.Linear(d, 3 * d, bias=False)
        self.proj = nn.Linear(d, d, bias=False)
        self.max_len = max_len

    def forward(self, x, cache=None):
        B, T, D = x.shape
        H, Dh = self.n_heads, self.head_dim
        q, k, v = self.qkv(x).chunk(3, dim=-1)
        q = q.view(B, T, H, Dh).transpose(1, 2)
        k = k.view(B, T, H, Dh).transpose(1, 2)
        v = v.view(B, T, H, Dh).transpose(1, 2)

        if cache is not None:
            past_k, past_v = cache
            k = torch.cat([past_k, k], dim=2)
            v = torch.cat([past_v, v], dim=2)

        # SDPA handles the causal mask internally
        out = F.scaled_dot_product_attention(q, k, v, is_causal=(cache is None))
        out = out.transpose(1, 2).contiguous().view(B, T, D)
        return self.proj(out), (k, v)


layer = CausalSelfAttention(d=512, n_heads=8).to(device).half()

# Simulate autoregressive generation
B, T_prompt = 1, 32
x_prompt = torch.randn(B, T_prompt, 512, device=device, dtype=torch.float16)

# Prefill
out, cache = layer(x_prompt)
print(f"After prefill ({T_prompt} tokens): "
      f"K shape = {tuple(cache[0].shape)}, V shape = {tuple(cache[1].shape)}")

# Decode 8 tokens, one at a time
for step in range(8):
    next_tok = torch.randn(B, 1, 512, device=device, dtype=torch.float16)
    out, cache = layer(next_tok, cache)
print(f"After 8 decode steps: K shape = {tuple(cache[0].shape)}")


**Observe.** During *prefill*, the cache is filled with `T_prompt` entries in a single forward pass. During *decode*, each forward pass adds exactly one entry. The compute profile of the two phases is radically different — prefill is *compute-bound* (a large GEMM over many tokens), decode is *memory-bound* (a tiny GEMM but a long KV-cache read). This asymmetry is why serving systems treat them separately.


## 5. vLLM: Production-Grade LLM Serving

vLLM (Kwon et al., 2023) is an open-source LLM serving framework built around PagedAttention. Its key features:

- **PagedAttention KV cache** — virtually unlimited concurrent requests, no fragmentation.
- **Continuous batching** — requests join and leave the batch at every step, rather than waiting for a fixed batch to complete.
- **Prefix caching** — identical prompt prefixes (system prompts, few-shot examples) share KV blocks.
- **Tensor parallelism** — automatic multi-GPU sharding.
- OpenAI-compatible HTTP API.

The code below assumes `vllm` is installed (`pip install vllm`) and that a small model fits in available memory. If you do not have the resources to run it, read through and proceed — the timing numbers are illustrative.


In [ ]:
# Example: serving a small model with vLLM (requires `pip install vllm`).
# This cell is intentionally guarded so the notebook completes even without vLLM.

import importlib.util
have_vllm = importlib.util.find_spec("vllm") is not None
print(f"vLLM available: {have_vllm}")

if have_vllm:
    from vllm import LLM, SamplingParams

    # Use a small model so the notebook is broadly runnable.
    # Replace with a larger model if you have the memory.
    llm = LLM(
        model="facebook/opt-125m",
        dtype="float16",
        gpu_memory_utilization=0.5,
        max_model_len=512,
    )

    sampling = SamplingParams(temperature=0.7, top_p=0.95, max_tokens=64)
    prompts = [
        "High-performance computing is",
        "The advantage of FlashAttention over naive attention is",
        "PagedAttention treats the KV cache like",
    ]
    outputs = llm.generate(prompts, sampling)
    for out in outputs:
        print(f"\n>>> {out.prompt!r}")
        print(f"    {out.outputs[0].text!r}")
else:
    print("Install vLLM to run this cell:  pip install vllm")


In [ ]:
# Throughput benchmark: many requests, mixed lengths.
if have_vllm:
    import random
    random.seed(0)

    prompts = [
        f"Write a short paragraph about {topic}." for topic in [
            "linear algebra", "neural networks", "GPU memory hierarchies",
            "the C programming language", "quantum field theory",
            "Byzantine architecture", "marine biology", "the Roman Senate",
        ] * 8
    ]
    sampling = SamplingParams(temperature=0.7, max_tokens=128)

    t0 = time.time()
    outs = llm.generate(prompts, sampling, use_tqdm=False)
    dt = time.time() - t0

    n_tokens = sum(len(o.outputs[0].token_ids) for o in outs)
    print(f"\nProcessed {len(prompts)} requests, generated {n_tokens} tokens.")
    print(f"Wall time   : {dt:.2f} s")
    print(f"Throughput  : {n_tokens / dt:.1f} tokens/s")
    print(f"Per request : {dt / len(prompts) * 1000:.1f} ms")
else:
    print("Skipping throughput benchmark (vLLM not installed).")


## 6. Exercises

**Exercise 3.1 — Memory accounting.**  
For a 13B-parameter dense transformer with $n_\ell = 40, H = 40, d = 128$, compute the KV-cache size per request at $L_{\text{ctx}} = 8192$ in FP16. How many concurrent requests fit in 80 GB after subtracting the weights and a 4 GB activation budget?

**Exercise 3.2 — Online softmax.**  
Implement the streaming softmax recurrence in NumPy. Verify it produces the same result as a one-shot softmax to within floating-point error.

**Exercise 3.3 — vLLM continuous batching.**  
Issue 64 requests of *random* length (`max_tokens` between 16 and 256). Compare total throughput to the case where every request has `max_tokens=256`. Explain the difference.

**Exercise 3.4 — Prefix caching.**  
Run two prompts that share the same 500-token system prefix. With `enable_prefix_caching=True`, measure the prefill latency of the second request and compare to the first.


## 7. Summary

Attention is memory-bound by construction. **FlashAttention** restructures the computation to compute outputs block-by-block, never materializing the $N \times N$ score matrix, and replaces quadratic HBM traffic with near-linear traffic. **PagedAttention** applies the operating-system insight of paged virtual memory to the KV cache, eliminating fragmentation and enabling production-grade serving systems like vLLM to achieve near-saturated GPU utilization.

Week 4 turns from *single-GPU* memory pressure to *multi-GPU* memory pressure: when a model's weights themselves exceed device memory, we need to shard them — and ZeRO and FSDP show us how.
